In [ ]:
import os
import json
import subprocess
from pathlib import Path
from typing import List, Optional, Dict
import getpass
import pandas as pd
from tqdm import tqdm
import time
from datetime import datetime

class VinDrMammoDownloader:
    """
    Production-grade VinDr-Mammo downloader optimized for Google Drive
    - Handles Drive mounting and verification
    - Implements robust error handling and retry logic
    - Provides checkpointing for interrupted downloads
    - Validates data integrity
    """

    def __init__(self, gdrive_path='/content/drive/MyDrive/vindr-mammo'):
        """
        Initialize downloader with Google Drive path

        Args:
            gdrive_path: Path within mounted Google Drive
        """
        self.base_dir = Path(gdrive_path)
        self.base_url = "https://physionet.org/files/vindr-mammo/1.0.0"
        self.progress_file = self.base_dir / 'download_progress.json'
        self.log_file = self.base_dir / 'download_log.txt'

        self.username = None
        self.password = None

        # Verify Google Drive access
        self._verify_gdrive_setup()

        # Create directory structure
        self.base_dir.mkdir(parents=True, exist_ok=True)
        (self.base_dir / 'images').mkdir(exist_ok=True)
        (self.base_dir / 'metadata').mkdir(exist_ok=True)
        (self.base_dir / 'logs').mkdir(exist_ok=True)

        self.progress = self._load_progress()
        self._log(f"Initialized downloader at {self.base_dir}")

    def _verify_gdrive_setup(self):
        """Verify Google Drive is properly mounted"""
        gdrive_root = Path('/content/drive')

        if not gdrive_root.exists():
            print("\n⚠️  Google Drive not mounted!")
            print("\n📌 Run this first:")
            print("   from google.colab import drive")
            print("   drive.mount('/content/drive')")
            raise RuntimeError("Google Drive not mounted")

        print("✅ Google Drive detected")

    def _log(self, message: str):
        """Log message to file and console"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_message = f"[{timestamp}] {message}"

        # Console output
        if "ERROR" in message or "❌" in message:
            print(f"❌ {message}")
        elif "WARNING" in message or "⚠️" in message:
            print(f"⚠️  {message}")

        # File logging
        try:
            log_file = self.base_dir / 'logs' / 'download_log.txt'
            with open(log_file, 'a') as f:
                f.write(log_message + '\n')
        except Exception as e:
            print(f"⚠️  Logging failed: {e}")

    def _load_progress(self) -> Dict:
        """Load download progress with validation"""
        if self.progress_file.exists():
            try:
                with open(self.progress_file, 'r') as f:
                    progress = json.load(f)
                print(f"📂 Loaded existing progress: {len(progress.get('downloaded_files', []))} files")
                return progress
            except Exception as e:
                self._log(f"ERROR loading progress, starting fresh: {e}")

        return {
            'downloaded_files': [],
            'failed_files': [],
            'total_size_mb': 0,
            'metadata_downloaded': False,
            'last_update': None,
            'dataset_structure': None
        }

    def _save_progress(self):
        """Save progress with atomic write"""
        self.progress['last_update'] = datetime.now().isoformat()

        # Atomic write: write to temp, then rename
        temp_file = self.progress_file.with_suffix('.tmp')
        try:
            with open(temp_file, 'w') as f:
                json.dump(self.progress, f, indent=2)
            temp_file.replace(self.progress_file)
        except Exception as e:
            self._log(f"ERROR saving progress: {e}")

    def setup_credentials(self, username: str = None, password: str = None) -> bool:
        """
        Setup and verify PhysioNet credentials

        Args:
            username: PhysioNet username
            password: PhysioNet password

        Returns:
            bool: True if credentials are valid
        """
        if not username:
            print("\n🔐 PhysioNet Credentials Required")
            print("   Get credentials at: https://physionet.org/")
            username = input("Username: ").strip()
            password = getpass.getpass("Password: ")

        self.username = username
        self.password = password

        print("🔍 Verifying credentials...")
        return self._test_access()

    def _test_access(self) -> bool:
        """Test PhysioNet access with credentials"""
        test_url = f"{self.base_url}/SHA256SUMS.txt"
        test_file = self.base_dir / "test_access.txt"

        cmd = [
            'wget',
            f'--user={self.username}',
            f'--password={self.password}',
            '-O', str(test_file),
            '-q', '--tries=2', '--timeout=15',
            test_url
        ]

        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=20)

            if result.returncode == 0 and test_file.exists():
                test_file.unlink()
                print("✅ Credentials verified!")
                return True
            else:
                print("❌ Authentication failed!")
                if result.stderr:
                    print(f"   Error: {result.stderr[:200]}")
                return False
        except subprocess.TimeoutExpired:
            print("❌ Connection timeout")
            return False
        except Exception as e:
            print(f"❌ Error: {e}")
            return False

    def download_metadata(self) -> bool:
        """Download CSV metadata files to Google Drive"""
        print("\n📊 Downloading Metadata Files")
        print("=" * 70)

        metadata_dir = self.base_dir / 'metadata'
        csv_files = [
            'breast-level_annotations.csv',
            'finding_annotations.csv'
        ]

        success = True
        for csv_file in csv_files:
            url = f"{self.base_url}/{csv_file}"
            output_file = metadata_dir / csv_file

            print(f"  📥 {csv_file}...", end=" ", flush=True)

            cmd = [
                'wget',
                f'--user={self.username}',
                f'--password={self.password}',
                '-O', str(output_file),
                '-q', '--timeout=60', '--tries=3',
                url
            ]

            try:
                result = subprocess.run(cmd, capture_output=True, timeout=90)

                if result.returncode == 0 and output_file.exists():
                    size_mb = output_file.stat().st_size / (1024 * 1024)
                    print(f"✅ ({size_mb:.2f} MB)")
                    self._log(f"Downloaded metadata: {csv_file}")
                else:
                    print("❌ Failed")
                    success = False
                    self._log(f"ERROR downloading {csv_file}")
            except Exception as e:
                print(f"❌ Error: {e}")
                success = False

        if success:
            self.progress['metadata_downloaded'] = True
            self._save_progress()
            print("\n✅ All metadata downloaded to Google Drive!\n")

        return success

    def build_file_list_from_metadata(self) -> Optional[List[str]]:
        """Build comprehensive file list from metadata"""
        print("\n📋 Building File List from Metadata...")

        csv_file = self.base_dir / 'metadata' / 'breast-level_annotations.csv'

        if not csv_file.exists():
            print("❌ Metadata not found. Downloading...")
            if not self.download_metadata():
                return None

        try:
            df = pd.read_csv(csv_file)

            print(f"  📊 Dataset statistics:")
            print(f"     Total annotations: {len(df)}")
            print(f"     Unique studies: {df['study_id'].nunique()}")
            print(f"     Unique images: {df['image_id'].nunique()}")

            # Build file paths
            file_list = []
            for _, row in df.iterrows():
                study_id = row['study_id']
                image_id = row['image_id']
                image_path = f"images/{study_id}/{image_id}.dicom"
                file_list.append(image_path)

            # Remove duplicates
            file_list = sorted(list(set(file_list)))

            print(f"\n  ✅ File list built: {len(file_list)} unique images")

            self.progress['file_list'] = file_list
            self._save_progress()

            return file_list

        except Exception as e:
            self._log(f"ERROR building file list: {e}")
            print(f"❌ Error: {e}")
            return None

    def download_by_study_batch(self, start_idx: int = 0, batch_size: int = 50) -> bool:
        """
        Download images by study batch (optimized for Google Drive)

        Args:
            start_idx: Starting study index
            batch_size: Number of studies to download

        Returns:
            bool: Success status
        """
        print(f"\n🎯 Downloading Studies {start_idx} to {start_idx + batch_size}")
        print("=" * 70)

        # Ensure metadata exists
        if not self.progress['metadata_downloaded']:
            if not self.download_metadata():
                return False

        # Read metadata
        csv_file = self.base_dir / 'metadata' / 'breast-level_annotations.csv'
        df = pd.read_csv(csv_file)

        # Get study batch
        all_studies = df['study_id'].unique()
        target_studies = all_studies[start_idx:start_idx + batch_size]

        print(f"📊 Total studies: {len(all_studies)}")
        print(f"🎯 Downloading: {len(target_studies)} studies\n")

        success_count = 0
        fail_count = 0
        downloaded_files = set(self.progress['downloaded_files'])

        for study_id in tqdm(target_studies, desc="Studies"):
            study_data = df[df['study_id'] == study_id]

            for _, row in study_data.iterrows():
                image_id = row['image_id']
                image_path = f"images/{study_id}/{image_id}.dicom"

                # Skip if already downloaded
                if image_path in downloaded_files:
                    success_count += 1
                    continue

                url = f"{self.base_url}/{image_path}"
                output_file = self.base_dir / image_path
                output_file.parent.mkdir(parents=True, exist_ok=True)

                # Download with retry
                if self._download_file(url, output_file):
                    success_count += 1
                    downloaded_files.add(image_path)
                    self.progress['downloaded_files'].append(image_path)
                else:
                    fail_count += 1
                    self.progress['failed_files'].append(image_path)

                # Periodic progress save (every 10 files)
                if (success_count + fail_count) % 10 == 0:
                    self._save_progress()

        # Final save
        self._save_progress()

        print(f"\n{'=' * 70}")
        print(f"✅ Batch Complete!")
        print(f"   Downloaded: {success_count}")
        print(f"   Failed: {fail_count}")
        print(f"   Total progress: {len(self.progress['downloaded_files'])} files")
        print(f"{'=' * 70}\n")

        return fail_count == 0

    def download_by_birads(self, birads_values: list = [4, 5, 6], batch_size: int = 50) -> bool:
        """
        Download images filtered by BIRADS categories (optimized for Google Drive)

        Args:
            birads_values: List of BIRADS categories to download (default: [4, 5, 6])
            batch_size: Number of files to download per batch for progress updates

        Returns:
            bool: Success status
        """
        print(f"\n🎯 Downloading BIRADS {birads_values} Images")
        print("=" * 70)

        # Ensure metadata exists
        if not self.progress['metadata_downloaded']:
            if not self.download_metadata():
                return False

        # Read metadata
        csv_file = self.base_dir / 'metadata' / 'breast-level_annotations.csv'

        try:
            df = pd.read_csv(csv_file)
        except Exception as e:
            print(f"❌ Error reading CSV: {e}")
            return False

        # Check if breast_birads column exists
        if 'breast_birads' not in df.columns:
            print(f"❌ Column 'breast_birads' not found!")
            print(f"   Available columns: {list(df.columns)}")
            return False

        # Debug: Show first few rows and unique values
        print(f"\n🔍 Debug Info:")
        print(f"   Total rows: {len(df)}")
        print(f"   Unique BIRADS values: {sorted(df['breast_birads'].dropna().unique())}")
        print(f"   Data type: {df['breast_birads'].dtype}")

        # Extract numeric values from "BI-RADS X" format
        # Handle formats like "BI-RADS 4", "BI-RADS 5", etc.
        df['birads_numeric'] = df['breast_birads'].str.extract(r'(\d+)')[0].astype(float)

        # Filter by BIRADS categories
        filtered_df = df[df['birads_numeric'].isin(birads_values)]

        print(f"\n📊 Total images in dataset: {len(df)}")
        print(f"🎯 BIRADS {birads_values} images: {len(filtered_df)}")

        if len(filtered_df) == 0:
            print(f"\n⚠️  WARNING: No images found with BIRADS values {birads_values}")
            print(f"   Available BIRADS values: {sorted(df['breast_birads'].dropna().unique())}")
            print(f"\n💡 Try one of these:")
            for val in sorted(df['breast_birads'].dropna().unique())[:5]:
                count = len(df[df['breast_birads'] == val])
                print(f"      downloader.download_by_birads([{val}])  # {count} images")
            return False

        print(f"📈 Percentage: {len(filtered_df)/len(df)*100:.1f}%\n")

        success_count = 0
        fail_count = 0
        downloaded_files = set(self.progress['downloaded_files'])

        # Process each filtered row
        for idx, row in tqdm(filtered_df.iterrows(), total=len(filtered_df), desc="Downloading"):
            study_id = row['study_id']
            image_id = row['image_id']
            birads = row['breast_birads']
            image_path = f"images/{study_id}/{image_id}.dicom"

            # Skip if already downloaded
            if image_path in downloaded_files:
                success_count += 1
                continue

            url = f"{self.base_url}/{image_path}"
            output_file = self.base_dir / image_path
            output_file.parent.mkdir(parents=True, exist_ok=True)

            # Download with retry
            if self._download_file(url, output_file):
                success_count += 1
                downloaded_files.add(image_path)
                self.progress['downloaded_files'].append(image_path)
            else:
                fail_count += 1
                self.progress['failed_files'].append(image_path)
                print(f"\n⚠️  Failed: {image_path} (BIRADS {birads})")

            # Periodic progress save (every batch_size files)
            if (success_count + fail_count) % batch_size == 0:
                self._save_progress()
                print(f"\n💾 Progress saved: {success_count} downloaded, {fail_count} failed")

        # Final save
        self._save_progress()

        print(f"\n{'=' * 70}")
        print(f"✅ Download Complete!")
        print(f"   Downloaded: {success_count}")
        print(f"   Failed: {fail_count}")
        print(f"   Total progress: {len(self.progress['downloaded_files'])} files")
        print(f"{'=' * 70}\n")

        return fail_count == 0

    def _download_file(self, url: str, output_file: Path, max_retries: int = 3) -> bool:
        """
        Download single file with retry logic

        Args:
            url: File URL
            output_file: Output path
            max_retries: Maximum retry attempts

        Returns:
            bool: Success status
        """
        for attempt in range(max_retries):
            cmd = [
                'wget',
                f'--user={self.username}',
                f'--password={self.password}',
                '-O', str(output_file),
                '-c', '-q',
                '--timeout=60',
                '--tries=2',
                url
            ]

            try:
                result = subprocess.run(cmd, capture_output=True, timeout=90)

                if result.returncode == 0 and output_file.exists() and output_file.stat().st_size > 0:
                    return True

                # Retry with backoff
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)

            except Exception as e:
                if attempt == max_retries - 1:
                    self._log(f"ERROR downloading {url}: {e}")

        return False

    def get_status(self):
        """Display comprehensive download status"""
        print("\n" + "=" * 70)
        print("📊 DOWNLOAD STATUS - VinDr-Mammo Dataset")
        print("=" * 70)
        print(f"📂 Location: {self.base_dir}")
        print(f"🗓️  Last update: {self.progress.get('last_update', 'Never')}")
        print(f"\n📈 Progress:")
        print(f"   ✅ Downloaded: {len(self.progress['downloaded_files'])} files")
        print(f"   ❌ Failed: {len(self.progress['failed_files'])} files")
        print(f"   📊 Metadata: {'✅' if self.progress['metadata_downloaded'] else '❌'}")

        # Calculate storage used
        if self.progress['downloaded_files']:
            total_size = sum(
                (self.base_dir / f).stat().st_size
                for f in self.progress['downloaded_files'][:100]  # Sample
                if (self.base_dir / f).exists()
            )
            avg_size_mb = total_size / len(self.progress['downloaded_files'][:100]) / (1024 * 1024)
            estimated_total = avg_size_mb * len(self.progress['downloaded_files'])
            print(f"   💾 Estimated size: {estimated_total:.2f} MB")

        print("=" * 70 + "\n")

    def retry_failed_downloads(self) -> bool:
        """Retry all failed downloads"""
        failed = self.progress['failed_files'].copy()

        if not failed:
            print("✅ No failed downloads to retry")
            return True

        print(f"\n🔄 Retrying {len(failed)} failed downloads...")

        self.progress['failed_files'] = []
        success = 0

        for file_path in tqdm(failed, desc="Retrying"):
            url = f"{self.base_url}/{file_path}"
            output_file = self.base_dir / file_path

            if self._download_file(url, output_file):
                success += 1
                self.progress['downloaded_files'].append(file_path)
            else:
                self.progress['failed_files'].append(file_path)

        self._save_progress()
        print(f"✅ Retry complete: {success}/{len(failed)} successful\n")
        return len(self.progress['failed_files']) == 0


# ==================== USAGE GUIDE ====================

def main():
    """
    Complete workflow for downloading VinDr-Mammo to Google Drive
    """

    print("🚀 VinDr-Mammo Downloader for Google Drive")
    print("=" * 70)

    # Step 1: Mount Google Drive (if not already mounted)
    print("\n📌 Step 1: Mount Google Drive")
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print("✅ Google Drive mounted")
    except:
        print("⚠️  Run in Google Colab or ensure Drive is mounted")

    # Step 2: Initialize downloader
    print("\n📌 Step 2: Initialize Downloader")
    downloader = VinDrMammoDownloader('/content/drive/MyDrive/vindr-mammo')

    # Step 3: Setup credentials
    print("\n📌 Step 3: Setup PhysioNet Credentials")
    if not downloader.setup_credentials():
        print("❌ Fix credentials and try again")
        return

    # Step 4: Download metadata
    print("\n📌 Step 4: Download Metadata")
    downloader.download_metadata()

    # Step 5: Download BIRADS 4-6 cancer cases
    print("\n📌 Step 5: Download BIRADS 4-6 Cases")
    downloader.download_by_birads([2])

    # Check status
    downloader.get_status()

    print("\n✅ Download session complete!")
    print("💡 Additional options:")
    print("   - Download other BIRADS: downloader.download_by_birads([1, 2, 3])")
    print("   - Download by study batch: downloader.download_by_study_batch(0, 50)")
    print("   - Retry failed: downloader.retry_failed_downloads()")

if __name__ == "__main__":
    main()

🚀 VinDr-Mammo Downloader for Google Drive

📌 Step 1: Mount Google Drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted

📌 Step 2: Initialize Downloader
✅ Google Drive detected
📂 Loaded existing progress: 1790 files

📌 Step 3: Setup PhysioNet Credentials

🔐 PhysioNet Credentials Required
   Get credentials at: https://physionet.org/
Username: dtobi59
Password: ··········
🔍 Verifying credentials...
✅ Credentials verified!

📌 Step 4: Download Metadata

📊 Downloading Metadata Files
  📥 breast-level_annotations.csv... ✅ (2.72 MB)
  📥 finding_annotations.csv... ✅ (3.33 MB)

✅ All metadata downloaded to Google Drive!


📌 Step 5: Download BIRADS 4-6 Cases

🎯 Downloading BIRADS [2] Images

🔍 Debug Info:
   Total rows: 20000
   Unique BIRADS values: ['BI-RADS 1', 'BI-RADS 2', 'BI-RADS 3', 'BI-RADS 4', 'BI-RADS 5']
   Data type: object

📊 Total images in dataset: 20000
🎯 BIRADS [2] images: 467

Downloading:  15%|█▍        | 700/4676 [29:09<81:33:45, 73.85s/it]


💾 Progress saved: 700 downloaded, 0 failed


Downloading:  16%|█▌        | 750/4676 [1:30:27<80:21:32, 73.69s/it]


💾 Progress saved: 750 downloaded, 0 failed


Downloading:  17%|█▋        | 800/4676 [2:32:16<81:28:38, 75.68s/it]


💾 Progress saved: 800 downloaded, 0 failed


Downloading:  18%|█▊        | 850/4676 [3:33:06<75:32:07, 71.07s/it]


💾 Progress saved: 850 downloaded, 0 failed


Downloading:  19%|█▉        | 877/4676 [4:07:13<82:06:19, 77.80s/it]